# FlyPose-SAR — CycleGAN RGB→Thermal (Fase 2)
## Setup prima di eseguire:
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → ON
3. Data → aggiungi: dataset_sar + llvip-dataset
4. Prima sessione: esegui celle 1→8 in ordine
5. Sessioni successive: esegui celle 1→5, poi cella RESUME, poi 7→8

## Configurazione: GroupNorm + batch=8 + 2×T4
- Ogni epoca: ~7 minuti (vs ~40 min con batch=1)
- 200 epoche totali: ~23h (~4 giorni con 2 sessioni/giorno)

## Cella 1 — Installazione

In [ ]:
!pip install torch torchvision Pillow tqdm -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


## Cella 2 — Trova dataset

In [ ]:
from pathlib import Path

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# Trova dataset_sar (dominio A)
dataset_sar_path = None
for d in KAGGLE_INPUT.rglob('dataset_sar'):
    if d.is_dir():
        dataset_sar_path = d
        break
if dataset_sar_path is None:
    for d in KAGGLE_INPUT.rglob('images'):
        if (d / 'train').exists() and (d / 'val').exists():
            dataset_sar_path = d.parent
            break

# Trova LLVIP infrared (dominio B)
llvip_path = None
for d in KAGGLE_INPUT.rglob('infrared'):
    train_dir = d / 'train'
    if train_dir.exists():
        imgs = list(train_dir.glob('*.jpg')) + list(train_dir.glob('*.png'))
        if len(imgs) > 100:
            llvip_path = str(train_dir)
            break

if dataset_sar_path is None:
    print('ERRORE: dataset_sar non trovato!')
else:
    print(f'[OK] Dataset SAR: {dataset_sar_path}')

if llvip_path is None:
    print('ERRORE: LLVIP non trovato!')
else:
    print(f'[OK] LLVIP thermal: {llvip_path}')

DIR_A = str(dataset_sar_path / 'images' / 'train')
DIR_B = llvip_path
print(f'\nDominio A (RGB)    : {DIR_A}')
print(f'Dominio B (Thermal): {DIR_B}')


## Cella 3 — Classi CycleGAN (GroupNorm)

In [ ]:
import torch
import torch.nn as nn
import random
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# GroupNorm compatibile con batch > 1
# num_groups=4 funziona con tutti i canali usati (64, 128, 256, 512)
def get_norm_layer(num_features, num_groups=4):
    return nn.GroupNorm(min(num_groups, num_features), num_features)

class ResidualBlock(nn.Module):
    def __init__(self, dim, norm_layer):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
            norm_layer(dim), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, kernel_size=3, padding=0, bias=True),
            norm_layer(dim),
        )
    def forward(self, x): return x + self.block(x)

class ResNetGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, n_blocks=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=True),
            get_norm_layer(ngf), nn.ReLU(inplace=True),
        ]
        for mult in [1, 2]:
            layers += [
                nn.Conv2d(ngf*mult, ngf*mult*2, kernel_size=3, stride=2, padding=1, bias=True),
                get_norm_layer(ngf*mult*2), nn.ReLU(inplace=True),
            ]
        for _ in range(n_blocks):
            layers.append(ResidualBlock(ngf*4, get_norm_layer))
        for mult in [4, 2]:
            layers += [
                nn.ConvTranspose2d(ngf*mult, ngf*mult//2, kernel_size=3, stride=2,
                                   padding=1, output_padding=1, bias=True),
                get_norm_layer(ngf*mult//2), nn.ReLU(inplace=True),
            ]
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0), nn.Tanh()]
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, input_nc=3, ndf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True)]
        nf = ndf
        for n in range(1, n_layers):
            nf_prev, nf = nf, min(nf*2, 512)
            layers += [nn.Conv2d(nf_prev, nf, kernel_size=4, stride=2, padding=1, bias=True),
                       get_norm_layer(nf), nn.LeakyReLU(0.2, inplace=True)]
        nf_prev, nf = nf, min(nf*2, 512)
        layers += [nn.Conv2d(nf_prev, nf, kernel_size=4, stride=1, padding=1, bias=True),
                   get_norm_layer(nf), nn.LeakyReLU(0.2, inplace=True),
                   nn.Conv2d(nf, 1, kernel_size=4, stride=1, padding=1)]
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class ImageBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.buffer   = []
    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.buffer) < self.max_size:
                self.buffer.append(img); result.append(img)
            else:
                if torch.rand(1).item() > 0.5:
                    idx = torch.randint(0, self.max_size, (1,)).item()
                    result.append(self.buffer[idx].clone()); self.buffer[idx] = img
                else:
                    result.append(img)
        return torch.cat(result, dim=0)

def init_weights(net, init_gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and ('Conv' in classname or 'Linear' in classname):
            nn.init.normal_(m.weight.data, 0.0, init_gain)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'GroupNorm' in classname:
            if m.weight is not None: nn.init.normal_(m.weight.data, 1.0, init_gain)
            if m.bias   is not None: nn.init.constant_(m.bias.data, 0.0)
    net.apply(init_func)
    return net

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

def get_transforms(img_size=256, augment=True):
    load_size = int(img_size * 1.12)
    if augment:
        return T.Compose([T.Resize(load_size, interpolation=T.InterpolationMode.BICUBIC),
                          T.RandomCrop(img_size), T.RandomHorizontalFlip(),
                          T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
    return T.Compose([T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC),
                      T.CenterCrop(img_size), T.ToTensor(),
                      T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

class UnpairedDataset(Dataset):
    def __init__(self, dir_A, dir_B, img_size=256, augment=True, max_images_b=5000):
        self.transform = get_transforms(img_size, augment)
        self.paths_A = sorted([p for p in Path(dir_A).rglob('*') if p.suffix.lower() in IMG_EXTENSIONS])
        self.paths_B = sorted([p for p in Path(dir_B).rglob('*') if p.suffix.lower() in IMG_EXTENSIONS])
        if max_images_b and len(self.paths_B) > max_images_b:
            step = len(self.paths_B) // max_images_b
            self.paths_B = self.paths_B[::step][:max_images_b]
            print(f'  Subsample B: 1 ogni {step} frame')
        self.size = max(len(self.paths_A), len(self.paths_B))
        print(f'  Dominio A (RGB):     {len(self.paths_A)} immagini')
        print(f'  Dominio B (Thermal): {len(self.paths_B)} immagini')
    def _load(self, path):
        img = Image.open(path)
        if img.mode != 'RGB': img = img.convert('RGB')
        return self.transform(img)
    def __len__(self): return self.size
    def __getitem__(self, idx):
        return {'A': self._load(self.paths_A[idx % len(self.paths_A)]),
                'B': self._load(self.paths_B[random.randint(0, len(self.paths_B)-1)])}

print('[OK] Classi caricate correttamente — GroupNorm + batch=8 + 2xT4')


## Cella 4 — Configurazione

In [ ]:
# ---- PARAMETRI ----
IMG_SIZE        = 256
BATCH_SIZE      = 8      # 4 per GPU × 2 GPU T4
N_EPOCHS        = 100    # epoche con lr costante
N_EPOCHS_DECAY  = 100    # epoche con lr decay (totale: 200)
LR              = 0.0002
BETA1           = 0.5
LAMBDA_CYCLE    = 10.0
LAMBDA_IDENTITY = 5.0
N_WORKERS       = 4
SAVE_FREQ       = 5
MAX_IMAGES_B    = 5000
# -------------------

# Multi-GPU: usa tutte le GPU disponibili
device       = 'cuda' if torch.cuda.is_available() else 'cpu'
n_gpus       = torch.cuda.device_count()
run_dir      = KAGGLE_WORKING / 'cyclegan_run'
total_epochs = N_EPOCHS + N_EPOCHS_DECAY
run_dir.mkdir(parents=True, exist_ok=True)

print(f'Device        : {device}')
print(f'GPU disponibili: {n_gpus}')
print(f'Batch totale  : {BATCH_SIZE} ({BATCH_SIZE//max(n_gpus,1)} per GPU)')
print(f'Epoche totali : {total_epochs}')
print(f'Tempo/epoca ~ : ~{20420 // (BATCH_SIZE * max(n_gpus,1)) * 3 // 60} min')
print(f'Run dir       : {run_dir}')


## Cella 5 — Dataloader

In [ ]:
print('[*] Caricamento dataset...')
dataset    = UnpairedDataset(DIR_A, DIR_B, IMG_SIZE, augment=True, max_images_b=MAX_IMAGES_B)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=N_WORKERS, pin_memory=True, drop_last=True)
print(f'[OK] {len(dataloader)} batch per epoca')


## Cella 6 — Inizializza modelli (SOLO PRIMA SESSIONE)
> Nelle sessioni successive usa la cella RESUME

In [ ]:
import itertools
from torch.optim import lr_scheduler

# Crea modelli e avvolgi con DataParallel per multi-GPU
G_AB = nn.DataParallel(init_weights(ResNetGenerator(3, 3))).to(device)
G_BA = nn.DataParallel(init_weights(ResNetGenerator(3, 3))).to(device)
D_A  = nn.DataParallel(init_weights(PatchGANDiscriminator(3))).to(device)
D_B  = nn.DataParallel(init_weights(PatchGANDiscriminator(3))).to(device)

criterion_gan      = nn.MSELoss()
criterion_cycle    = nn.L1Loss()
criterion_identity = nn.L1Loss()

opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                            lr=LR, betas=(BETA1, 0.999))
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=(BETA1, 0.999))

def lambda_rule(epoch):
    return 1.0 - max(0, epoch - N_EPOCHS) / float(N_EPOCHS_DECAY + 1)
sched_G   = lr_scheduler.LambdaLR(opt_G,   lr_lambda=lambda_rule)
sched_D_A = lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lambda_rule)
sched_D_B = lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lambda_rule)

buffer_A    = ImageBuffer(50)
buffer_B    = ImageBuffer(50)
start_epoch = 1
history     = {'epoch':[], 'G':[], 'D_A':[], 'D_B':[], 'cycle':[], 'identity':[], 'lr':[]}

# Conta parametri sul modello base (senza DataParallel wrapper)
params_G = sum(p.numel() for p in G_AB.module.parameters())/1e6
params_D = sum(p.numel() for p in D_B.module.parameters())/1e6
print(f'[OK] G_AB: {params_G:.1f}M parametri')
print(f'[OK] D_B:  {params_D:.1f}M parametri')
print(f'[OK] DataParallel su {n_gpus} GPU')
print('[OK] Pronto per la prima sessione')


## Cella RESUME — Riprendi da checkpoint (SESSIONI 2, 3, 4...)
> Esegui questa cella AL POSTO della cella 6
> Aggiorna RESUME_CHECKPOINT con il percorso del tuo ultimo checkpoint

In [ ]:
import itertools
from torch.optim import lr_scheduler

# ---- AGGIORNA QUESTO ----
RESUME_CHECKPOINT = '/kaggle/input/<nome-dataset-checkpoint>/checkpoint_epoch005.pth'
# -------------------------

# Ricrea modelli con DataParallel
G_AB = nn.DataParallel(ResNetGenerator(3, 3)).to(device)
G_BA = nn.DataParallel(ResNetGenerator(3, 3)).to(device)
D_A  = nn.DataParallel(PatchGANDiscriminator(3)).to(device)
D_B  = nn.DataParallel(PatchGANDiscriminator(3)).to(device)

criterion_gan      = nn.MSELoss()
criterion_cycle    = nn.L1Loss()
criterion_identity = nn.L1Loss()

opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                            lr=LR, betas=(BETA1, 0.999))
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=(BETA1, 0.999))

def lambda_rule(epoch):
    return 1.0 - max(0, epoch - N_EPOCHS) / float(N_EPOCHS_DECAY + 1)
sched_G   = lr_scheduler.LambdaLR(opt_G,   lr_lambda=lambda_rule)
sched_D_A = lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lambda_rule)
sched_D_B = lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lambda_rule)

buffer_A = ImageBuffer(50)
buffer_B = ImageBuffer(50)

# Carica checkpoint — i pesi sono stati salvati con .module per compatibilità
ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
G_AB.module.load_state_dict(ckpt['G_AB'])
G_BA.module.load_state_dict(ckpt['G_BA'])
D_A.module.load_state_dict(ckpt['D_A'])
D_B.module.load_state_dict(ckpt['D_B'])
history     = ckpt['history']
start_epoch = ckpt['epoch'] + 1

# Avanza scheduler alla posizione corretta
for _ in range(ckpt['epoch']):
    sched_G.step(); sched_D_A.step(); sched_D_B.step()

print(f'[OK] Ripreso da epoca {ckpt["epoch"]}')
print(f'[OK] Prossima epoca  : {start_epoch}')
print(f'[OK] Epoche rimanenti: {total_epochs - ckpt["epoch"]}')
print(f'[OK] DataParallel su {n_gpus} GPU')


## Cella 7 — Training loop

In [ ]:
import json

for epoch in range(start_epoch, total_epochs + 1):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    e_G = e_DA = e_DB = e_cyc = e_idt = 0.0
    n   = 0

    for batch in dataloader:
        real_A = batch['A'].to(device)
        real_B = batch['B'].to(device)

        # Generatori
        for p in D_A.parameters(): p.requires_grad_(False)
        for p in D_B.parameters(): p.requires_grad_(False)
        opt_G.zero_grad()
        idt_A  = G_AB(real_B); loss_idt_A = criterion_identity(idt_A, real_B) * LAMBDA_IDENTITY
        idt_B  = G_BA(real_A); loss_idt_B = criterion_identity(idt_B, real_A) * LAMBDA_IDENTITY
        fake_B = G_AB(real_A); fake_A = G_BA(real_B)
        loss_G_AB = criterion_gan(D_B(fake_B), torch.ones_like(D_B(fake_B)))
        loss_G_BA = criterion_gan(D_A(fake_A), torch.ones_like(D_A(fake_A)))
        rec_A = G_BA(fake_B); loss_cyc_A = criterion_cycle(rec_A, real_A) * LAMBDA_CYCLE
        rec_B = G_AB(fake_A); loss_cyc_B = criterion_cycle(rec_B, real_B) * LAMBDA_CYCLE
        loss_G = loss_G_AB + loss_G_BA + loss_cyc_A + loss_cyc_B + loss_idt_A + loss_idt_B
        loss_G.backward(); opt_G.step()

        # Discriminatore B
        for p in D_B.parameters(): p.requires_grad_(True)
        opt_D_B.zero_grad()
        pred_r  = D_B(real_B)
        loss_DB = (criterion_gan(pred_r, torch.ones_like(pred_r)) +
                   criterion_gan(D_B(buffer_B.push_and_pop(fake_B.detach())),
                                 torch.zeros_like(pred_r))) * 0.5
        loss_DB.backward(); opt_D_B.step()

        # Discriminatore A
        for p in D_A.parameters(): p.requires_grad_(True)
        opt_D_A.zero_grad()
        pred_r  = D_A(real_A)
        loss_DA = (criterion_gan(pred_r, torch.ones_like(pred_r)) +
                   criterion_gan(D_A(buffer_A.push_and_pop(fake_A.detach())),
                                 torch.zeros_like(pred_r))) * 0.5
        loss_DA.backward(); opt_D_A.step()

        e_G  += loss_G.item(); e_DB += loss_DB.item()
        e_DA += loss_DA.item()
        e_cyc += (loss_cyc_A + loss_cyc_B).item()
        e_idt += (loss_idt_A + loss_idt_B).item()
        n += 1
        if n % 500 == 0:
            print(f'  step {n}/{len(dataloader)} | G={e_G/n:.3f} D_B={e_DB/n:.3f}')

    sched_G.step(); sched_D_A.step(); sched_D_B.step()
    lr = opt_G.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{total_epochs} | '
          f'G={e_G/n:.3f} D_A={e_DA/n:.3f} D_B={e_DB/n:.3f} | '
          f'Cyc={e_cyc/n:.3f} Idt={e_idt/n:.3f} | lr={lr:.6f}')

    history['epoch'].append(epoch)
    history['G'].append(e_G/n);    history['D_A'].append(e_DA/n)
    history['D_B'].append(e_DB/n); history['cycle'].append(e_cyc/n)
    history['identity'].append(e_idt/n); history['lr'].append(lr)

    if epoch % SAVE_FREQ == 0 or epoch == total_epochs:
        ckpt_path = run_dir / f'checkpoint_epoch{epoch:03d}.pth'
        # Salva i pesi senza il wrapper DataParallel (.module)
        # così il checkpoint è compatibile con qualsiasi configurazione GPU
        torch.save({
            'epoch': epoch,
            'G_AB':  G_AB.module.state_dict(),
            'G_BA':  G_BA.module.state_dict(),
            'D_A':   D_A.module.state_dict(),
            'D_B':   D_B.module.state_dict(),
            'history': history,
        }, ckpt_path)
        print(f'  [SAVE] {ckpt_path.name}')


## Cella 8 — Salva zip
> Esegui prima di chiudere la sessione

In [ ]:
import zipfile, json

# Salva pesi finali senza DataParallel wrapper
torch.save(G_AB.module.state_dict(), run_dir / 'G_AB_final.pth')
torch.save(G_BA.module.state_dict(), run_dir / 'G_BA_final.pth')
with open(run_dir / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

zip_path = KAGGLE_WORKING / 'cyclegan_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['G_AB_final.pth', 'G_BA_final.pth', 'training_history.json']:
        p = run_dir / fname
        if p.exists():
            zf.write(p, fname)
            print(f'  [+] {fname}')
    for ckpt in sorted(run_dir.glob('checkpoint_*.pth')):
        zf.write(ckpt, ckpt.name)
        print(f'  [+] {ckpt.name}')

print(f'\n[OK] ZIP: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)')
print('Scarica da: Output (pannello destro) → cyclegan_results.zip')


In [ ]:
import shutil
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

# Carica G_AB_final (no DataParallel)
G_AB_gen = ResNetGenerator(3, 3).to(device)
G_AB_gen.load_state_dict(torch.load(
    '/kaggle/input/datasets/attimatti/flypose-cyclegan-v2-ckpt/G_AB_final.pth',
    map_location=device))
G_AB_gen.eval()
print('[OK] G_AB caricato')

src_img = Path(DIR_A)
src_lbl = Path(LABELS_A)
out_img = Path('/kaggle/working/dataset_sar_thermal/images/train')
out_lbl = Path('/kaggle/working/dataset_sar_thermal/labels/train')
out_img.mkdir(parents=True, exist_ok=True)
out_lbl.mkdir(parents=True, exist_ok=True)

tf = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

with torch.no_grad():
    for i, p in enumerate(sorted(src_img.glob('*.jpg'))):
        img = Image.open(p).convert('RGB')
        t   = tf(img).unsqueeze(0).to(device)
        out = G_AB_gen(t)[0].cpu() * 0.5 + 0.5
        T.ToPILImage()(out.clamp(0,1)).save(out_img / p.name)
        lbl = src_lbl / (p.stem + '.txt')
        if lbl.exists():
            shutil.copy(lbl, out_lbl / lbl.name)
        if i % 1000 == 0:
            print(f'{i}/22671')

print('[DONE] Generazione completata')